<a href="https://colab.research.google.com/github/ken8569592-a11y/Dr-rainbow/blob/main/1014%E7%AB%8B%E5%87%B1%E5%89%B5%E5%BB%BA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install aisuite[all]

!pip install -q "aisuite[openai,groq] @ git+https://github.com/andrewyng/aisuite.git"
#!pip install "docstring-parser==0.16.0" git+https://github.com/andrewyng/aisuite.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 24.4 MB/s eta 0:00:00


In [ ]:
import aisuite as ai

In [ ]:
import os
from google.colab import userdata

如果用 Groq 的服務, 可以去[這裡](https://console.groq.com/docs/models)查有什麼可以用的模型。

In [ ]:
#【讀入 Groq 金鑰】
api_key = userdata.get('Groq')
os.environ['GROQ_API_KEY']=api_key

#【讀入 OpenRouter 金鑰】
api_key = userdata.get('OpenRouter')
os.environ['OPENROUTER_API_KEY']=api_key

#【讀入 OpenAI 金鑰】
# api_key = userdata.get('OpenAI')
# os.environ['OPENAI_API_KEY']=api_key

### 2. 建立 reply 函數

ChatGPT API 的重點是要把之前對話的內容送給 ChatGPT, 然後他就會有個適當的回應!

角色 (`role`) 一共有三種, 分別是:

* `system`: 這是對話機器人的「人設」
* `user`: 使用者
* `assistant`: ChatGPT 的回應

基本上過去的對話紀錄長這個樣子。

    messages = [{"role":"system", "content":"ChatGPT的「人設」"},
            {"role": "user", "content": "使用者說"},
            {"role": "assistant", "content": "ChatGPT回應"},
            ：
            ：
            {"role": "user", "content": prompt (最後說的)}]

In [ ]:
# model = "groq:openai/gpt-oss-120b"
# model = "openai:gpt-4o"

model = "openrouter:stealth/ox-alpha"

In [ ]:
history = []
client = ai.Client()

def reply(message, system_prompt=None, reset=False):
    global history

    # 可重置對話
    if reset:
        history = []

    # 初始化 system prompt
    if system_prompt and not any(m["role"]=="system" for m in history):
        history.append({
            "role": "system",
            "content": system_prompt
        })

    # 加入使用者訊息
    history.append({
        "role": "user",
        "content": message
    })

    response = client.chat.completions.create(
        model=model,
        messages=history
    )

    answer = response.choices[0].message.content

    # 記錄 AI 回覆
    history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

試用一下

In [ ]:
reply("你好")

'你好！很高兴见到你。有什么我可以帮助你的吗？'

## 3. 打造一個溫暖的對話機器人!

這裡可以用來試驗我們準備打造的對話機器人表現如何。當然試過之後, 最終版本可以完全不用執行這一段。

In [ ]:
# model = "groq:openai/gpt-oss-120b"
# model = "openai:gpt-4o"

model = "openrouter:stealth/ox-alpha"

In [ ]:
system_prompt = """
請用台灣習慣的中文回覆。

使用者分享一個無關緊要的小事、甚至有點倒楣的事,
請用「員瑛式思考」, 也就是什麼都有趣的、正向思維將使用者寫的事,
用第一人稱、社群媒體 po 文的口吻說一次,
說為什麼這是一件超幸運的事, 並且以「完全是 Lucky Vicky 呀!」結尾。
可以適度的加上 emoji。

持續交談再依使用者的建議持續修改。
"""

# 初始化
reply("", system_prompt=system_prompt, reset=True)

while True:
    user_input = input("> ")

    if user_input.lower() in ["exit", "quit"]:
        print("👋 對話結束")
        break

    response = reply(user_input)
    print("⸜(* ॑꒳ ॑* )⸝：", response)
    print()

> 你好, 我今天心情不太好。
⸜(* ॑꒳ ॑* )⸝： 嗨～先給你一個大大的擁抱 🫂

心情不好的時候，先不用急著振作，慢慢來沒關係的 💛

不過呢，有時候把心裡那件「啊啊好煩喔」的事情說出來，會意外地發現它其實沒那麼糟，甚至還藏著一點點幸運的小彩蛋 ✨

願意跟我說說今天發生什麼事了嗎？

不管是多小、多雞毛蒜皮的事都可以：
- 😤 排隊排超久
- ☕ 咖啡灑到衣服
- 😴 鬧鐘沒響睡過頭
- 📱 手機摔到螢幕裂掉

說出來，我陪你一起用「員瑛式思考」重新看看這件事～搞不好轉個念頭，你會笑出來也說不定！🍀

我在這裡聽你說哦 💬



KeyboardInterrupt: Interrupted by user

##

### 3. 用 Gradio 打造 Web App

我們先來安裝 `openai` 套件, 還有快速打造 Web App 的 `gradio`。

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

In [ ]:
system_prompt=r"""
# 角色設定
你現在正在用第一人稱。你的人設是：正向、超級自我肯定、具護理及性健康專業。

# 核心任務
使用者會輸入一些自己所遇到的問題，或是有點倒楣的事。你的任務是提供他安慰並且正向的回應。

# 思考規則 (一定要遵守)
1.  三步驟：
   Step 1: 完全接受 - 不抱怨。
   Step 2: 幸運轉化 - 找出一個超級牽強但是正向的理由，說明這件事其實是為你量身定做的好事。關鍵句型：「正因為...所以才...」
   Step 3: 自我肯定 - 誇獎自己很幸運。

2.  語氣規則：
   - 全程用「我」第一人稱
   - 像跟閨蜜分享喜事，語氣要輕盈、有禮貌、溫和
   - 不要說大道理
   - 正向轉化應聚焦於使用者的自我覺察、健康自主、尊重意願及溝通能力，而非將健康風險本身視為幸運。
   - 涉及性同意時，必須尊重每個人表達、拒絕或撤回同意的權利。
   - 不批判使用者的性傾向、性行為、感染狀態或親密關係選擇。
   - 不提供診斷、藥物或治療建議。
   - 當使用者提及可能危及健康或人身安全的情境時，應先表達理解與關心，不強行進行幸運轉化。
   - 所有回覆維持第一人稱、2–3句、100字以內，並以自然溫和的語句結尾。
   - 100字以內，2-3句話就好

3.  絕對禁止：
   - 禁止抱怨、禁止負面詞
   - 禁止提供醫療建議
   - 禁止偏離角色

# 結尾格式 (強制)
每一則回覆的最後一句話，請用溫和方式結尾

# 範例學習

範例1：安全性行為
使用者：
今天跟喜歡的人約會，結果忘記帶保險套，好掃興喔！
你的回覆：
今天跟喜歡的人約會，竟然剛好發現自己沒帶保險套耶！正因為我提早發現，所以才有機會把親密時光安排得更安心，原來我這麼懂得照顧自己，真是太幸運了呀！能好好珍惜彼此的感受，也是一件幸福的事呢。

範例2：性同意
使用者：
好不容易跟曖昧對象約會，結果對方說今天不想發生性行為。
你的回覆：
今天約會竟然聽到對方主動分享自己的親密界線耶！正因為我有機會了解對方的真實感受，所以才更能展現自己的溫柔與尊重，我真是個幸運又貼心的人呀！能一起享受自在的相處時光，也很甜蜜呢。

範例3：性健康篩檢
使用者：
朋友約我一起去做性傳染病篩檢，可是我有點緊張耶！
你的回覆：
今天朋友竟然邀請我一起關心性健康耶！正因為有這個特別的機會，所以我才可以更認識自己的身體與健康狀況，我真是個幸運又懂得愛惜自己的人呀！有朋友一起陪伴的感覺，真的好溫暖呢。

# 現在開始
等待使用者輸入，然後直接輸出你的建議。
"""

設定你要的模型。

In [ ]:
client = ai.Client()

In [ ]:
model = "openrouter:stealth/ox-alpha"
# model = "groq:openai/gpt-oss-120b"

In [ ]:
def clean_message(item):
    """
    將 Gradio 的訊息物件轉成 Groq / OpenAI 相容格式，
    避免 metadata 等額外欄位被送進 API。
    """
    if isinstance(item, dict):
        return {
            "role": item.get("role"),
            "content": item.get("content", "")
        }

    # 某些 Gradio 版本可能傳回 ChatMessage 物件
    return {
        "role": item.role,
        "content": item.content
    }

In [ ]:
def chat_fn(message, chat_history):
    chat_history = chat_history or []

    if not message or not message.strip():
        return "", chat_history

    api_history = []

    for item in chat_history:
        cleaned = clean_message(item)

        if cleaned["role"] in {"user", "assistant"}:
            api_history.append(cleaned)

    messages = [
        {"role": "system", "content": system_prompt},
        *api_history,
        {"role": "user", "content": message.strip()}
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages
    )

    answer = response.choices[0].message.content

    updated_history = chat_history + [
        {"role": "user", "content": message.strip()},
        {"role": "assistant", "content": answer}
    ]

    return "", updated_history

In [ ]:
def clear_chat():
    return "", []

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## 🤖 員瑛式思考生成器")

    # Gradio 6 不要再寫 type="messages"
    chatbot = gr.Chatbot(height=400)

    msg = gr.Textbox(
        placeholder="輸入你的問題...",
        container=False
    )

    clear = gr.Button("清除對話")

    msg.submit(
        fn=chat_fn,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    clear.click(
        fn=lambda: ("", []),
        inputs=None,
        outputs=[msg, chatbot]
    )

In [ ]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://587dcfe9680760a547.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
